## Overview

**Conv2d Layer Decomposition** uses Tucker decomposition to factorize convolutional layers into three smaller, more efficient convolutions. This is the Conv2d counterpart of FC Decomposition (which uses SVD for Linear layers).

### How It Works

A Conv2d weight tensor $W \in \mathbb{R}^{C_{out} \times C_{in} \times H \times W}$ is decomposed into three convolutions:

1. `Conv2d(C_in, R_in, 1)` — pointwise input compression
2. `Conv2d(R_in, R_out, (H, W))` — spatial convolution at reduced rank
3. `Conv2d(R_out, C_out, 1)` — pointwise output expansion

Where $R_{in}$ and $R_{out}$ are the Tucker ranks, controlled by `percent_removed`.

### When to Use Conv Decomposition

| Model Type | Conv Layer Size | Recommendation |
|------------|-----------------|----------------|
| ResNet-style | Medium 3×3 convolutions | ✅ **Effective** — 2-4x FLOP reduction |
| VGG-style | Large 3×3 convolutions | ✅ **Highly effective** |
| MobileNet | Already uses depthwise separable | ❌ Skipped (grouped convolutions) |
| 1×1 convolutions | Pointwise | ❌ Skipped automatically |

**Key advantage:** Works on any hardware — no sparse kernel requirements.

In [ ]:
#| include: false
from fastai.vision.all import *
from fasterai.misc.all import *
import torch
import torch.nn as nn

## 1. Setup and Data

In [ ]:
path = untar_data(URLs.PETS)
files = get_image_files(path/"images")
def label_func(f): return f[0].isupper()
dls = ImageDataLoaders.from_name_func(path, files, label_func, item_tfms=Resize(64))

## 2. Train the Model

We use ResNet-18 — a Conv-heavy model where Tucker decomposition is effective:

In [ ]:
learn = vision_learner(dls, resnet18, metrics=accuracy)
learn.unfreeze()
learn.fit(3)

## 3. Apply Tucker Decomposition

Use `Conv_Decomposer` to factorize all eligible Conv2d layers:

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

original_params = count_params(learn.model)
print(f"Original parameters: {original_params:,}")

# Decompose — remove 50% of rank per mode
decomposer = Conv_Decomposer()
new_model = decomposer.decompose(learn.model, percent_removed=0.5)

new_params = count_params(new_model)
print(f"Decomposed parameters: {new_params:,}")
print(f"Reduction: {(1 - new_params/original_params)*100:.1f}%")
print(f"Compression: {original_params/new_params:.1f}x")

### What Happened?

Each eligible Conv2d layer (kernel > 1×1, not grouped) was replaced by a Sequential of 3 smaller convolutions:

```
Before: Conv2d(64, 128, 3×3)           — 73,728 parameters
After:  Conv2d(64, 32, 1×1)            — 2,048 parameters
        Conv2d(32, 64, 3×3)            — 18,432 parameters
        Conv2d(64, 128, 1×1)           — 8,192 parameters
                                         28,672 parameters (2.6x smaller)
```

1×1 convolutions and depthwise convolutions are skipped automatically.

## 4. Accuracy Before Fine-Tuning

In [ ]:
new_learn = Learner(dls, new_model, metrics=accuracy)
new_learn.validate()

The accuracy drops after decomposition — Tucker decomposition is an **approximation**, not exact. Fine-tuning recovers most of the accuracy:

```python
new_learn.fit_one_cycle(3, 1e-4)  # Fine-tune with small learning rate
```

## 5. Controlling Compression

The `percent_removed` parameter controls how much rank is removed per mode:

| percent_removed | Rank Kept | Compression | Accuracy Impact |
|-----------------|-----------|-------------|-----------------|
| `0.0` | 100% | ~1x (near-exact) | Minimal |
| `0.3` | 70% | ~1.5-2x | Low |
| `0.5` | 50% | ~2-4x | Moderate |
| `0.7` | 30% | ~4-8x | Significant |

```python
# Light compression — minimal accuracy loss
light = Conv_Decomposer().decompose(model, percent_removed=0.3)

# Heavy compression — needs fine-tuning
heavy = Conv_Decomposer().decompose(model, percent_removed=0.7)
```

## 6. Combining with Other Techniques

Tucker decomposition works well as a first step before other compressions:

```python
from fasterai.misc.all import Conv_Decomposer, BN_Folder

# 1. Fold BatchNorm into Conv layers
model = BN_Folder().fold(model)

# 2. Decompose Conv layers
model = Conv_Decomposer().decompose(model, percent_removed=0.5)

# 3. Fine-tune
learn = Learner(dls, model, metrics=accuracy)
learn.fit_one_cycle(3, 1e-4)

# 4. Quantize for deployment
from fasterai.quantize.quantizer import Quantizer
model = Quantizer(backend='x86', method='static').quantize(model, dls.valid)
```

### Recommended ordering:
```
BN Fold → Tucker Decompose → Fine-tune → Prune → Quantize
```

---

## Summary

| Metric | ResNet-18 (50% removed) |
|--------|------------------------|
| Original Params | ~11.7M |
| Decomposed Params | ~5-7M |
| Compression | ~1.7-2.3x |
| Accuracy (before fine-tune) | Drops ~10-20% |
| Accuracy (after fine-tune) | Recovers to within 1-3% |

| Feature | Description |
|---------|-------------|
| `Conv_Decomposer()` | Create a decomposer instance |
| `.decompose(model, percent_removed)` | Decompose all eligible Conv2d layers |
| Skips 1×1 convolutions | Already minimal — decomposition would increase params |
| Skips grouped convolutions | Tucker assumes standard convolution |
| Pure PyTorch | No external dependencies (no tensorly) |

---

## See Also

- [FC Decomposer](tutorial.fc_decomposer.html) - SVD decomposition for Linear layers
- [BN Folding](bn_folding.html) - Fold BatchNorm before decomposition
- [Pruner Tutorial](../prune/pruner.html) - Apply after decomposition for further compression